# Prevalence and properness: what changes when the labels do

Two questions about a score: does it change when the class mix changes, and does it reward honest
probabilities? This notebook replicates the positives of each label of *yeast* and records which
summaries move, then tabulates which scoring rules are proper and checks that, on 0/1 scores, the
calibration errors are one minus recall and one minus specificity.

In [1]:
# Installs the package only where it is missing (for example in Colab).
import importlib.util
if importlib.util.find_spec("aggmetrics") is None:
    %pip install -q "aggmetrics>=0.3" scikit-learn matplotlib pandas

In [2]:
import io
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from aggmetrics.scores import ScoreMatrix

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10,
})

## Data and scores

In [3]:
URL = "https://raw.githubusercontent.com/rcprati/aggmetrics/main/examples/data/yeast.npz"
local = Path("../data/yeast.npz")
source = local if local.exists() else io.BytesIO(urllib.request.urlopen(URL, timeout=60).read())
with np.load(source) as z:
    X, Y = z["X"], z["Y"].astype(int)
print("examples, features, labels:", X.shape[0], X.shape[1], Y.shape[1])

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    "kNN (k=10)": make_pipeline(StandardScaler(), KNeighborsClassifier(10)),
}

def out_of_fold_scores(model):
    """Scores in [0, 1] for every example, each from a model that did not see it."""
    P = np.zeros(Y.shape)
    for train, test in KFold(5, shuffle=True, random_state=0).split(X):
        fitted = MultiOutputClassifier(model).fit(X[train], Y[train])
        P[test] = np.column_stack([p[:, 1] for p in fitted.predict_proba(X[test])])
    return P

scores = {name: ScoreMatrix(Y, out_of_fold_scores(m)) for name, m in models.items()}

examples, features, labels: 2417 103 14


## Replicating positives

`skew_report` repeats the positive examples of each label (here twice as often) and records the
change of each summary.

In [4]:
from aggmetrics.scores import skew_report

sm = scores["Random forest"]
r = skew_report(sm, factor=2)
rows = [{"summary": s, "largest change, own label": np.nanmax(np.abs(r["own"][s]))}
        for s in ("alpha", "beta", "auc", "ndcg", "brier")]
pd.DataFrame(rows).set_index("summary").round(4)

,"largest change, own label"
summary,
alpha,0.0000
beta,0.0000
auc,0.0000
ndcg,0.0000
brier,0.0644


`alpha_j`, `beta_j`, `AUC_j` and per-class nDCG do not move when the label's own positives are
replicated: they are conditional on the class. The per-class Brier score does, because it mixes
`alpha_j` and `beta_j` with the prevalence. Replicating rows also changes the *other* labels that
share those examples:

In [5]:
pd.DataFrame({s: [np.nanmax(np.abs(r["others"][s]))] for s in ("alpha", "beta", "auc", "ndcg")},
             index=["largest change, other labels"]).round(4)

,alpha,beta,auc,ndcg
"largest change, other labels",0.0343,0.011,0.0261,0.0206


Because yeast examples carry several labels, replicating the rows of one label moves the scores of
the others; the report says by how much. Global summaries move too:

In [6]:
for key, val in r["globals"].items():
    print(f"{key:14s}", np.round(np.nanmax(np.abs(val)), 4))

brier_micro    0.0107
auc_macro      0.0126
auc_micro      0.0135
ndcg_micro     0.0331
ndcg_macro     0.0032
ndcg_exemplar  0.0439


## Which scoring rules are proper

A rule is proper if reporting the true probability is optimal. `properness_table` tests the
deviation operators numerically and states the ranking ones.

In [7]:
from aggmetrics.scores import properness_table

pd.DataFrame(properness_table())[["name", "family", "proper", "strictly_proper"]]

,name,family,proper,strictly_proper
0,absolute,deviation,False,False
1,log,deviation,True,True
2,squared,deviation,True,True
3,dcg,ranking,True,False
4,ndcg,ranking,False,False
5,precision_at_k,ranking,True,False
6,auc,ranking,None,False


Brier and log loss are strictly proper; absolute error is not (it rewards pushing scores to 0 or 1).
Raw DCG is only weakly proper, nDCG is not proper in general, and AUC is proper only under
conditions (`proper` is `None`): they ignore the magnitude of the scores, so an order-preserving
distortion leaves them unchanged.

## The hard case

With scores in `{0, 1}` the squared deviation of a cell is its error indicator, so `alpha_j = 1 -
recall_j` and `beta_j = 1 - specificity_j`. `hard_bridge` checks this against the indicator matrices
used in the rest of the package.

In [8]:
from aggmetrics.scores import hard_bridge

h = hard_bridge(sm, threshold=0.5)
print("consistent with IndicatorMatrix:", h["consistent"])
print("alpha_j      :", np.round(h["alpha"][:5], 3), "...")
print("1 - recall_j :", np.round(1 - h["recall"][:5], 3), "...")

consistent with IndicatorMatrix: True
alpha_j      : [0.58  0.529 0.381 0.56  0.639] ...
1 - recall_j : [0.58  0.529 0.381 0.56  0.639] ...


## What to take away

- Use `alpha_j`, `beta_j`, per-class AUC or per-class nDCG to compare labels or datasets with
  different class mixes; per-class Brier, micro AUC and exemplar nDCG move with prevalence.
- Prefer a proper rule (Brier, log loss) when the scores are meant to be probabilities.
- The hard measures of the rest of the package are the special case of 0/1 scores.